# Práctica 1 — Clasificación y Comparativa de Algoritmos
**Materia:** Aprendizaje Automático  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Comparar el rendimiento de cinco algoritmos clásicos de clasificación (KNN, SVM, Árbol de Decisión, Random Forest, Gradient Boosting) sobre el dataset Breast Cancer Wisconsin.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
print("Librerías importadas.")

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print(f"Muestras: {X.shape[0]} | Características: {X.shape[1]}")
print(f"Clases: {data.target_names} | Distribución: {np.bincount(y)}")
df = pd.DataFrame(X, columns=data.feature_names)
df.describe().round(2)

In [ ]:
sc = StandardScaler()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

## 2. Entrenamiento y evaluación

In [ ]:
modelos = {
    'KNN (k=5)':      KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)':      SVC(kernel='rbf', C=10, probability=True, random_state=42),
    'Árbol':          DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':  RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boost': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

resultados = []
for nombre, clf in modelos.items():
    clf.fit(X_train_s, y_train)
    acc  = clf.score(X_test_s, y_test)
    cv   = cross_val_score(clf, X_train_s, y_train, cv=5, scoring='accuracy').mean()
    auc  = roc_auc_score(y_test, clf.predict_proba(X_test_s)[:,1])
    resultados.append({'Modelo': nombre, 'Acc. prueba': acc, 'CV-5 acc': cv, 'AUC-ROC': auc})
    print(f"{nombre:20s} Acc: {acc:.4f} | CV: {cv:.4f} | AUC: {auc:.4f}")

df_res = pd.DataFrame(resultados).sort_values('AUC-ROC', ascending=False)
print("\n", df_res.round(4).to_string(index=False))

## 3. Curvas ROC comparativas

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
for nombre, clf in modelos.items():
    RocCurveDisplay.from_estimator(clf, X_test_s, y_test, ax=ax, name=nombre)
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_title('Curvas ROC — Breast Cancer Wisconsin')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('roc_curves.png', dpi=120); plt.show()

## 4. Importancia de características (Random Forest)

In [ ]:
rf = modelos['Random Forest']
importancias = pd.Series(rf.feature_importances_, index=data.feature_names).sort_values(ascending=False)[:15]
importancias.plot(kind='barh', figsize=(8,5), color='steelblue')
plt.title('Top 15 características — Random Forest')
plt.xlabel('Importancia'); plt.gca().invert_yaxis()
plt.tight_layout(); plt.savefig('feature_importance.png', dpi=120); plt.show()

## Conclusiones
- Gradient Boosting y Random Forest obtienen el mejor AUC-ROC al combinar múltiples estimadores débiles.
- SVM con kernel RBF es competitivo pero más sensible a la escala de los datos (requiere normalización).
- El análisis de importancia de características revela que el radio y el perímetro medio son los predictores más relevantes.